# Explore the ODC-TBI Harmonized Preclinical Dataset with `mlcroissant`

## Introduction
This dataset, from Gold et al. (2024), provides a harmonized, multi-site collection of diffusion tensor imaging (DTI) MRI data from a preclinical rat model of traumatic brain injury (TBI). It addresses a key challenge in neuroimaging: the variability introduced by different imaging sites and acquisition protocols. Using the NeuroCombat harmonization technique, the authors corrected for site effects, enabling more robust and reproducible analyses of TBI-related brain changes.

Learn more:
- FAIR² Data Package: [10.71728/senscience.g7x2-a9k4](https://sen.science/doi/10.71728/senscience.g7x2-a9k4)

As a FAIR² Data Package, it ensures accessibility, interoperability, and AI-readiness. FAIR² datasets follow the MLCommons **Croissant** 🥐 format for machine learning datasets. See the [MLCommons Croissant Format Specification](https://docs.mlcommons.org/croissant/docs/croissant-spec.html).

This notebook provides a step-by-step guide for loading the dataset using the `mlcroissant` Python library and recreating some of the key analyses from the data article.

### Install and import required libraries

In [ ]:
# Install mlcroissant and the analysis dependencies
!pip install mlcroissant==1.0.22
!pip install seaborn==0.13.2 tabulate==0.9.0 plotly==6.1.2

In [ ]:
import mlcroissant as mlc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tabulate import tabulate
from IPython.display import Markdown, display

## 1. Data Loading
Load metadata and records from the dataset. This notebook uses a local copy of the Croissant metadata file (`fair2.json`), which in turn points to the local data file (`odc-tbi_1342.csv`). **Please ensure both files are in the same directory as this notebook.**

In [ ]:
# Provide the local path to the dataset's metadata file
croissant_file_path = 'https://sen.science/doi/10.71728/senscience.g7x2-a9k4/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_file_path)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview

In the **Croissant** format, a `RecordSet` represents a structured collection of records. It defines the structure of these records using a set of fields, such as the columns in our CSV file.

### 2.1 Review available RecordSets

In [ ]:
# List all the record sets available in the dataset
df_record_sets = pd.DataFrame(metadata["recordSet"])
columns_to_keep = {
    "@id": "Record Set ID",
    "description": "Description"
}
df_record_sets = df_record_sets[list(columns_to_keep.keys())]
df_record_sets = df_record_sets.rename(columns=columns_to_keep)

# Convert DataFrame to Markdown table
markdown_table = tabulate(df_record_sets, headers="keys", tablefmt="pipe", showindex=False)

# Render the table as Markdown in Jupyter
display(Markdown(markdown_table))

## 3. Data Extraction and Preparation

We will load the data from the primary record set into a pandas DataFrame for analysis.

In [ ]:
record_set_id = "https://sen.science/doi/10.71728/senscience.g7x2-a9k4/recordsets/odc-tbi_1342"
prefix = "https://sen.science/doi/10.71728/senscience.g7x2-a9k4/record-sets/odc-tbi_1342/fields/"
df = pd.DataFrame(dataset.records(record_set_id))
df.columns = [col.replace(prefix, "") if col.startswith(prefix) else col for col in df.columns]
# mlcroissant gives text values as bytes: decode them once, as strings
df = df.apply(lambda column: column.map(lambda v: v.decode('utf-8') if isinstance(v, bytes) else v))

# Display the first few rows and schema of the dataframe
display(Markdown(f"### First 5 rows of the `{record_set_id}` RecordSet"))
display(df.head())

display(Markdown("### DataFrame Information"))
df.info()

The dataframe contains subject identifiers, experimental variables (`location`: the site; `injgroupassigntyp`: the group, `CCI` or `SHM`; `sextyp`; `injelapsedtime`: `d03` or `d30` days after injury), and volumes of brain tissue with significantly high or low fractional anisotropy (FA), mean (MD), axial (AD) and radial (RD) diffusivity. The `unharmonized_` volumes are the raw ones, the `harmonized_` and `voxel_harmonized_` ones are harmonized (without values for the UF site), and the FA volumes ending in `_4site` are harmonized across the four sites.

## 4. Exploratory Data Analysis (EDA)

Now we will perform an EDA to understand the dataset's structure and replicate key findings from the data article.

### 4.1 Study Design Overview
First, let's summarize the distribution of subjects across the different experimental factors, as shown in Figure 2 of the data article.

In [ ]:
display(Markdown("#### Subject Counts by Site and Group"))
design_summary = df.groupby(['location', 'injgroupassigntyp'])['subjectid'].nunique().unstack()
display(design_summary)

display(Markdown("#### Subject Counts by Timepoint and Group"))
timepoint_summary = df.groupby(['injelapsedtime', 'injgroupassigntyp'])['subjectid'].nunique().unstack()
display(timepoint_summary)

In [ ]:
# Convert index and columns from bytes to strings for plotting
def decode_bytes(val):
    return val.decode() if isinstance(val, bytes) else val

design_summary_plot = design_summary.copy()
design_summary_plot.index = [decode_bytes(idx) for idx in design_summary_plot.index]
design_summary_plot.columns = [decode_bytes(col) for col in design_summary_plot.columns]

timepoint_summary_plot = timepoint_summary.copy()
timepoint_summary_plot.index = [decode_bytes(idx) for idx in timepoint_summary_plot.index]
timepoint_summary_plot.columns = [decode_bytes(col) for col in timepoint_summary_plot.columns]

# Plot the design_summary table as a grouped bar chart
fig1, ax1 = plt.subplots(figsize=(8, 5))
design_summary_plot.plot(kind='bar', ax=ax1)
ax1.set_title('Subject Counts by Site and Group')
ax1.set_xlabel('Site')
ax1.set_ylabel('Number of Subjects')
ax1.legend(title='Group')
plt.tight_layout()
plt.show()

# Plot the timepoint_summary table as a grouped bar chart
fig2, ax2 = plt.subplots(figsize=(6, 4))
timepoint_summary_plot.plot(kind='bar', ax=ax2)
ax2.set_title('Subject Counts by Timepoint and Group')
ax2.set_xlabel('Timepoint (Days Post-Injury)')
ax2.set_ylabel('Number of Subjects')
ax2.legend(title='Group')
plt.tight_layout()
plt.show()

### 4.2 Analysis of Data Harmonization

A key contribution of this dataset is the harmonization of data across the four sites, which reduces site-specific variance that could otherwise confound analysis.

Let's visualize this effect on the volume of tissue with high fractional anisotropy (`FA`), harmonized across the four sites (`_4site` columns), in the sham group, where differences between sites are not due to the injury.

In [ ]:
# We will focus on the sham group to see baseline site differences
sham_df = df[df['injgroupassigntyp'] == 'SHM']

# Set up the figure for side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Plot 1: Before Harmonization
sns.boxplot(ax=axes[0], data=sham_df, x='location', y='unharmonized_high_fa_volume_4site', hue='location',
            palette='viridis', legend=False)
axes[0].set_title('Before Harmonization (Raw)')
axes[0].set_xlabel('Site')
axes[0].set_ylabel('Volume of high-FA tissue')

# Plot 2: After Harmonization
sns.boxplot(ax=axes[1], data=sham_df, x='location', y='harmonized_high_fa_volume_4site', hue='location',
            palette='viridis', legend=False)
axes[1].set_title('After Harmonization (NeuroCombat, four sites)')
axes[1].set_xlabel('Site')
axes[1].set_ylabel('') # Remove redundant y-axis label

fig.suptitle('Effect of NeuroCombat Harmonization on Site Differences (Sham Group)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In the sham group, the raw volumes of high-FA tissue differ markedly between the four sites (left plot). After NeuroCombat harmonization across the four sites, the site distributions are much closer (right plot), indicating that site-specific technical variability has been reduced.

### 4.3 Analysis of TBI Effects Over Time

With the harmonized data, we can investigate the biological effects of TBI. Let's examine the harmonized volume of tissue with high radial diffusivity (`RD`), comparing the TBI group (`CCI`) to the control group (`SHM`) at the two timepoints, 3 and 30 days after injury.

In [ ]:
plt.figure(figsize=(12, 7))
sns.pointplot(
    data=df,
    x='injelapsedtime',
    y='harmonized_high_rd_volume',
    hue='injgroupassigntyp',
    palette={'SHM': 'royalblue', 'CCI': 'darkorange'},
    dodge=True,
    errorbar='se'
)

plt.title('Effect of TBI on the Volume of Tissue with High Radial Diffusivity', fontsize=16)
plt.xlabel('Timepoint (d03: 3 days, d30: 30 days post-injury)')
plt.ylabel('Harmonized volume of high-RD tissue')
plt.legend(title='Group')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


The volume of tissue with high radial diffusivity (RD) is much larger in the TBI (`CCI`) group than in the sham controls at both timepoints, and it increases from 3 to 30 days after injury in the TBI group, while it stays stable in the sham group. This is consistent with the known pathophysiology of TBI, where increased RD is thought to reflect demyelination and axonal damage, and with an injury that progresses over the first month.

### 4.4 Correlation of DTI Metrics

Finally, let's explore the relationships between the volumes of tissue with high values of the four DTI metrics in the harmonized data.

In [ ]:
# Select the four key DTI metrics for the corpus callosum from the harmonized data
dti_metrics_df = df[[
    'harmonized_high_fa_volume',
    'harmonized_high_md_volume',
    'harmonized_high_ad_volume',
    'harmonized_high_rd_volume'
]].copy()

# Rename columns for better readability in the plot
dti_metrics_df.columns = ['High FA', 'High MD', 'High AD', 'High RD']

# Calculate the correlation matrix
corr_matrix = dti_metrics_df.corr()

# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Correlation of the Volumes of Tissue with High DTI Metrics', fontsize=16)
plt.show()

The volumes of tissue with high mean (`MD`), axial (`AD`) and radial (`RD`) diffusivity are strongly and positively correlated with each other, as expected since MD is the mean of AD and RD. The volume of tissue with high fractional anisotropy (`FA`), a measure of directional coherence, is essentially uncorrelated with them.

## 5. Observations

1.  **Dataset Structure**: The dataset provides a comprehensive table of preclinical TBI data, including subject-level information, experimental groups, and a rich set of DTI metrics for numerous brain regions, with both raw and harmonized values available.

2.  **Data Harmonization**: The NeuroCombat harmonization across the four data collection sites (`_4site` columns) markedly reduced the differences between sites in the sham group, which is crucial for pooling data in multi-center studies.

3.  **Biological Insights**: The harmonized data allowed for a clear visualization of TBI's impact. The volume of tissue with high Radial Diffusivity (`RD`), a key indicator of white matter damage, is much larger after TBI, and increases from 3 to 30 days after injury.

4.  **Data Integrity**: The strong, expected correlations between the volumes of tissue with high MD, AD and RD support the internal consistency of the imaging data.

## Conclusion

In this notebook, we successfully loaded and explored the ODC-TBI Harmonized Preclinical Dataset using the `mlcroissant` library. By loading the data via its FAIR² Croissant description, we were able to quickly access and structure it for analysis. Our exploratory analysis replicated several key findings from the associated data article, demonstrating the effectiveness of the data harmonization and the clear biological signals related to traumatic brain injury. This dataset is a valuable, AI-ready resource that can power more robust and reproducible research, ultimately aiding in the discovery and validation of TBI biomarkers.